# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lilitkarapetyanofficial/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring.**

My lane is **Refresh / Content Opportunity Scoring**: given a client's existing pages, produce
an ordered queue of which pages a reviewer should look at first. The decision it improves is
"which of a client's existing pages should be reviewed first this cycle, given limited review
capacity?" — a reviewer never has time to check every page, so what they need is an ordering
with a cutoff, not a single yes/no verdict per page.

It is tempting to call this classification, since under the hood the pipeline trains a binary
classifier on `is_declining_label`. But the *output that gets used* is the model's predicted
probability turned into a **score** used to **rank** pages within a client — the binary label
is a means to build the score, not the deliverable. That distinction matters for the metric too
(Section 3): a ranking problem is judged on whether the right pages land near the top
(precision@K), not on overall accuracy across all 30,000 pages, most of which nobody will ever
look at this cycle.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from pathlib import Path

def find_data_csv(filename="content_refresh_anonymized.csv"):
    here = Path.cwd()
    for base in [here, *here.parents]:
        candidate = base / "data" / "raw" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find data/raw/{filename} above {here}.")

CSV_PATH = find_data_csv()
df = pd.read_csv(CSV_PATH)

print("Rows x columns:", df.shape)
print("Distinct clients (queues are ranked WITHIN a client):", df["client_id"].nunique())
print()
print("A reviewer's review capacity is finite and per-client -- e.g. review the top 20")
print("pages per client this cycle, not the top 20 pages globally, since clients differ")
print("in portfolio size:")
print(df.groupby("client_id").size().describe()[["min", "mean", "max"]].round(1))


Rows x columns: (30000, 44)
Distinct clients (queues are ranked WITHIN a client): 32

A reviewer's review capacity is finite and per-client -- e.g. review the top 20
pages per client this cycle, not the top 20 pages globally, since clients differ
in portfolio size:
min        3.0
mean     937.5
max     7008.0
dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`**, a binary flag defined in the pipeline as
`is_declining_label = (trend_direction == "down")`.

This is a **defined-rule label, not a freshly observed outcome** — I need to be honest about
that rather than dress it up. `trend_direction` itself is computed from `trend_pct`, a measured
90-day trend in the underlying GSC/GA4 metrics. So the label is one step removed from a raw
rule (like "flag if not updated in 180 days"): it's derived from *measured* traffic/position
movement, not from someone's opinion or an arbitrary threshold on content age. That's better
than a hand-picked rule, but it is still a **computed proxy for the real target I care about**,
which is "editorial attention would help this page." A page can be declining because it truly
needs a refresh, or because the query it targets simply lost seasonal demand — the label can't
tell those apart, and neither can I from this data alone.

Two consequences follow directly, and both are load-bearing for every later step:
- Because the label is *derived from* `trend_direction`/`trend_pct`, those two columns must
  **never** be used as model features — using them would mean the model just re-learns its own
  label definition (leakage), not the underlying pattern.
- Because it's a proxy and not ground truth, I can only claim the model *predicts the proxy
  label well* — not that it "detects true content decline" or "knows what needs a refresh."


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Label source chain: trend_pct (measured) -> trend_direction (down/up/flat) -> is_declining_label")
print()
print("trend_direction value counts:")
print(df["trend_direction"].value_counts())
print()
print("Declining-label rate:", round(df["is_declining_label"].mean(), 3))
print()
print("Columns that must be excluded from features because they define the label:")
print(["trend_direction", "trend_pct"])


Label source chain: trend_pct (measured) -> trend_direction (down/up/flat) -> is_declining_label

trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining-label rate: 0.542

Columns that must be excluded from features because they define the label:
['trend_direction', 'trend_pct']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: precision@50** — of the top 50 pages the model ranks highest (roughly a reviewer's
capacity for one cycle across a mid-size client set in this slice), what fraction are actually
`is_declining_label == True`?

I'm choosing precision@K over overall accuracy or ROC-AUC as the *headline* number because the
decision only ever touches the top of the queue — a reviewer with limited time will never see
row 25,000, so a metric that rewards getting the *bottom* of the ranking right too (like
accuracy) doesn't reflect the real cost function. Precision@K asks exactly the question the
reviewer cares about: "of the pages I'm about to spend my limited time on, how many were worth
it?" I'll still report ROC-AUC and recall as supporting numbers, but precision@K is the one I'd
defend if asked "did this work?"

I can compute this today on a trivial baseline (rank by `days_since_last_update` alone) and
compare it to the repo's committed rule baseline and trained model, so the metric is real and
not defined after the fact.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

naive_rank = df.sort_values("days_since_last_update", ascending=False).head(50)
naive_precision_at_50 = naive_rank["is_declining_label"].mean()
print(f"Naive baseline (rank by staleness alone), precision@50: {naive_precision_at_50:.3f}")

print()
print("From outputs/model_report.md (same dataset, full rule baseline vs trained model):")
print("  baseline_rules   precision@50 = 0.240")
print("  random_forest    precision@50 = 0.740")
print()
print("'Good' for this task: comfortably beating the rule baseline's 0.240 -- the model")
print("report's 0.740 is the number this framing is aiming to reproduce and defend.")


Naive baseline (rank by staleness alone), precision@50: 0.520

From outputs/model_report.md (same dataset, full rule baseline vs trained model):
  baseline_rules   precision@50 = 0.240
  random_forest    precision@50 = 0.740

'Good' for this task: comfortably beating the rule baseline's 0.240 -- the model
report's 0.740 is the number this framing is aiming to reproduce and defend.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one content page** (`content_id`), scored and ranked *within*
its client (`client_id`). Not a client, not a day, not a keyword — the review decision a
reviewer makes is page by page, so the row the model scores has to match that.

Below is the actual slice: identifiers, the label, and the features I would build the score
from (excluding the label-defining columns from Section 2).


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unit_of_analysis_cols = [
    "content_id", "client_id",           # identifiers -- grouping only, never features
    "is_declining_label",                  # target
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "days_since_last_update", "content_age_days",
    "word_count", "content_type",
]

sample = df[unit_of_analysis_cols].sample(5, random_state=42)
print("One row = one content page. Shape of full slice:", df[unit_of_analysis_cols].shape)
sample


One row = one content page. Shape of full slice: (30000, 11)


,content_id,client_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,content_age_days,word_count,content_type
2308,content_9824710082d8,client_3fdba35f04,0,283,0,0.00,21.3,104,174,1397.0,keyword article
22404,content_3efa3a7c46bb,client_f74efabef1,0,8878,8,0.09,8.8,20,134,3188.0,keyword article
23397,content_575dc8a2ab0f,client_25fc0e7096,0,3,0,0.00,0.0,8,109,3381.0,feedly article
25058,content_0dbd6911ba04,client_d029fa3a95,1,124,0,0.00,8.1,20,151,2892.0,comparison article
2664,content_bbaf87019afb,client_19581e27de,1,4294,10,0.23,30.9,22,466,NaN,keyword article


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A plain rule sounds reasonable at first: "flag any page not updated in 180+ days." The numbers
below show why that's too blunt to trust with limited review capacity. The signal that actually
separates a declining page from a healthy one is spread across several correlated fields —
traffic volume, position, freshness, content age, content type — and they interact rather than
add up cleanly. A page can be perfectly fresh and still be declining because its target query
lost demand; another can be stale for a year and be perfectly healthy because it never had much
traffic to lose. A single if-statement threshold can't represent that kind of conditional,
many-signal pattern, but it's exactly the kind of tangled-but-real pattern a model is built to
pick up — which is also why the repo's own committed baseline-vs-model comparison shows the
rule reaching only 0.240 precision@50 against the model's 0.740 on this same data.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show concretely why a single-field staleness rule misses most of the real
# decliners, and why a decline can happen even on freshly-updated pages.

stale_rule = df["days_since_last_update"] >= 180
n_declining = df["is_declining_label"].sum()

caught_by_rule = df.loc[stale_rule, "is_declining_label"].sum()
missed_by_rule = n_declining - caught_by_rule

fresh_but_declining = df[(df["days_since_last_update"] < 90) & (df["is_declining_label"] == 1)]

print(f"Total declining pages: {n_declining}")
print(f"Caught by 'stale >= 180 days' rule: {caught_by_rule} "
      f"({round(100 * caught_by_rule / n_declining, 1)}%)")
print(f"Missed by that same rule: {missed_by_rule} "
      f"({round(100 * missed_by_rule / n_declining, 1)}%)")
print()
print(f"Pages updated within the last 90 days that are STILL declining: {len(fresh_but_declining)}")
print("-> freshness alone doesn't predict decline in either direction; the rule is blunt")
print("   in both directions, which is exactly what a many-signal model can untangle.")


Total declining pages: 16262
Caught by 'stale >= 180 days' rule: 82 (0.5%)
Missed by that same rule: 16180 (99.5%)

Pages updated within the last 90 days that are STILL declining: 10576
-> freshness alone doesn't predict decline in either direction; the rule is blunt
   in both directions, which is exactly what a many-signal model can untangle.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.